# A fisher flow matching example for text


In [1]:
!git clone https://github.com/SchulzKilian/flow_matching.git
%cd flow_matching


fatal: destination path 'flow_matching' already exists and is not an empty directory.
/content/flow_matching


In [2]:
!git pull



Already up to date.


In [3]:
!pip install -e .



Obtaining file:///content/flow_matching
  Preparing metadata (setup.py) ... done
  Attempting uninstall: flow_matching
    Found existing installation: flow_matching 1.0.10
    Uninstalling flow_matching-1.0.10:
      Successfully uninstalled flow_matching-1.0.10
  Running setup.py develop for flow_matching


## Imports and init device

In [4]:
%ls

assets/             data_cache/      flow_matching/           RELEASE.md
CHANGELOG.md        docs/            flow_matching.egg-info/  setup.py
CODE_OF_CONDUCT.md  environment.yml  LICENSE                  tests/
CONTRIBUTING.md     examples/        README.md


In [5]:
import time
import torch
import math
import numpy as np
import sys
import os
from torch import nn, Tensor
import torch.nn.functional as F


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

text_path = os.path.abspath("examples/text")

if text_path not in sys.path:
    sys.path.append(text_path)



# flow_matching
from examples.text.model.transformer import Transformer
# from examples.text.data import DataState
from examples.text.data.data import get_data_state, get_data_loaders
from flow_matching.path import GeodesicProbPath
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.solver import ODESolver, RiemannianODESolver
from flow_matching.utils import ModelWrapper
from flow_matching.utils.manifolds import Sphere, Manifold


# visualization
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from matplotlib import cm

In [6]:
if torch.cuda.is_available():
    device = 'cuda:0'
    print('Using gpu')
else:
    device = 'cpu'
    print('Using cpu.')

Using gpu


In [7]:
torch.manual_seed(42)

## Dataset

In [8]:
def inf_train_gen(batch_size: int = 200, device: str = "cpu"):
    x1 = torch.rand(batch_size, device=device) * 4 - 2
    x2_ = (torch.rand(batch_size, device=device) - torch.randint(high=2, size=(batch_size, ), device=device) * 2)
    x2 = x2_ + (torch.floor(x1) % 2)

    data = torch.cat([x1[:, None], x2[:, None]], dim=1)

    return data.float()

def wrap(manifold, samples):
    center = torch.cat([torch.zeros_like(samples), torch.ones_like(samples[..., 0:1])], dim=-1)
    samples = torch.cat([samples, torch.zeros_like(samples[..., 0:1])], dim=-1) / 2

    return manifold.expmap(center, samples)

## Model

In [9]:
conf = OmegaConf.create({
    # --- Model Params (Used by transformer.py) ---
    "hidden_size": 384,
    "n_heads": 6,
    "cond_dim": 64,          # Dimension for time embeddings
    "dropout": 0.1,
    "n_blocks": 4,
    "model": { "length": 16 }, # Sequence Length (keep small for testing)

    # --- Data Params (Used by data.py) ---
    "data": {
        "train": "wikitext103",
        "valid": "wikitext103",
        "cache_dir": "./data_cache",
        "num_workers": 2
    },
    "training": { "batch_size": 128 },
    "compute": { "ngpus": 1 },
    "eval": { "batch_size": 128 }
})

# Parameters derived from config
seq_len = conf.model.length
vocab_size = 50304 # We will clip real data to this size for this demo
dim = seq_len * vocab_size # Total dimension of the flattened sphere vector


In [10]:


class ProjectToTangent(nn.Module):
    """Projects a vector field onto the tangent plane at the input."""

    def __init__(self, vecfield: nn.Module, manifold: Manifold):
        super().__init__()
        self.vecfield = vecfield
        self.manifold = manifold

    def forward(self, x: Tensor, t: Tensor) -> Tensor:
        x = self.manifold.projx(x)
        v = self.vecfield(x, t)
        v = self.manifold.proju(x, v)
        return v

class FlowMatchingTransformer(Transformer):
    def __init__(self, vocab_size, masked, config):
        # Initialize the original Transformer
        super().__init__(vocab_size, masked, config)

        # 1. OVERWRITE the embedding layer
        # Original was nn.Embedding (Integers -> Vector)
        # We need nn.Linear (Sphere Vector -> Vector)
        self.vocab_embed = nn.Linear(vocab_size, config.hidden_size)
        self.expected_flat_dim = config.model.length * vocab_size

    def forward(self, x, t):
      # 1. FIX TIME SHAPE: Ensure t is (Batch,), not (Batch, 1)
      if t.ndim > 1:
          t = t.squeeze()

      # 2. FIX DATA SHAPE: Strip appended time column if present
      if x.shape[-1] == self.expected_flat_dim + 1:
          x = x[:, :-1]

      # 3. Reshape x to Sequence: (Batch, SeqLen, Vocab)
      b_size = x.shape[0]
      x_reshaped = x.view(b_size, self.config.model.length, self.vocab_size)

      # 4. Run Transformer
      out = super().forward(x_reshaped, t)

      # 5. Flatten output
      return out.view(b_size, -1)

# --- ASSERTION: Verify Model Input/Output ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_model = FlowMatchingTransformer(vocab_size, False, conf).to(device)
test_input = torch.randn(32, dim).to(device) # Batch 32, Flat Dimension
test_time = torch.rand(32).to(device)

with torch.no_grad():
    test_out = test_model(test_input, test_time)

assert test_out.shape == test_input.shape, f"Shape mismatch! Expected {test_input.shape}, got {test_out.shape}"
print("✅ Model Wrapper is working. Input/Output shapes align.")

✅ Model Wrapper is working. Input/Output shapes align.


**Data Setup**

In [11]:
import torch.distributed as dist

# Running on a fake distributed system
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "12355" # Any free port

# 'gloo' works on both CPU and GPU and is safer for notebooks.
# rank=0 means "I am the main process".
# world_size=1 means "There is only 1 GPU total".
if not dist.is_initialized():
    dist.init_process_group(backend="gloo", rank=0, world_size=1)

In [12]:
import torch
from torch.utils.data import DataLoader, Dataset
import requests

# --- 1. Get Raw Data ---
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

# --- 2. Build Minimal Vocab ---
chars = sorted(list(set(text)))
vocab_size = len(chars) # ~65
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# --- 3. Simple Dataset Wrapper ---
class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = torch.tensor([stoi[c] for c in data], dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        return self.data[idx:idx+self.seq_len]

# --- 4. Update Config & Load ---
conf.model.length = 64  # Increase length since chars are small
conf.training.batch_size = 128

dataset = CharDataset(text, conf.model.length)
train_loader = DataLoader(dataset, batch_size=conf.training.batch_size, shuffle=True, drop_last=True)
train_iter = iter(train_loader)

print(f"✅ Data Ready. Vocab Size: {vocab_size}, Sequence Length: {conf.model.length}")

✅ Data Ready. Vocab Size: 65, Sequence Length: 64


In [13]:
import torch.nn.functional as F

# --- SAFE HELPER FUNCTION ---
# This defines the function but DOES NOT reset your train_loader
def get_text_batch(iterator, loader, vocab_size):
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(loader)
        batch = next(iterator)

    # Handle both Dict (Library) and Tensor (Custom) cases automatically
    if isinstance(batch, dict):
        # Fallback if you ever switch back to library data
        ids = list(batch.values())[0]
    else:
        # This is what your Tiny Shakespeare loader uses
        ids = batch

    ids = ids.to(device)

    # Safety Check
    if ids.max() >= vocab_size:
        raise ValueError(f"CRITICAL ERROR: Data index {ids.max()} > Vocab Size {vocab_size}.")

    # One-hot encode and flatten
    x = F.one_hot(ids, num_classes=vocab_size).float()
    return x.view(x.shape[0], -1), iterator

print("✅ Helper function defined. Data was NOT overwritten.")

✅ Helper function defined. Data was NOT overwritten.


In [14]:
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Create a folder for your models so they don't get lost
save_folder = "/content/drive/My Drive/Shakespeare_Flow_Models"
os.makedirs(save_folder, exist_ok=True)

print(f"✅ Checkpoints will be saved to: {save_folder}")

Mounted at /content/drive
✅ Checkpoints will be saved to: /content/drive/My Drive/Shakespeare_Flow_Models


## Train Velocity Flow Matching model

In [15]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import requests
import time

# --- 1. Global Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQ_LEN = 128
BATCH_SIZE = 128
LR = 1e-3

# --- 2. Fresh Data Setup (Character Level) ---
print("Downloading Tiny Shakespeare...")
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

# Create Vocab
chars = sorted(list(set(text)))
vocab_size = len(chars) # ~65
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
print(f"✅ Vocab Size: {vocab_size}")

# Create Dataset
class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = torch.tensor([stoi[c] for c in data], dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        return self.data[idx:idx+self.seq_len]

# Create Loader
dataset = CharDataset(text, SEQ_LEN)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
train_iter = iter(train_loader)

# --- 3. Model Setup ---
# Ensure Config Matches Data
conf.model.length = SEQ_LEN
conf.vocab_size = vocab_size
conf.training.batch_size = BATCH_SIZE

# Re-Initialize Model
manifold = Sphere()
vf = ProjectToTangent(
    FlowMatchingTransformer(
        vocab_size=vocab_size,
        masked=False,
        config=conf
    ),
    manifold=manifold,
)
vf.to(device)

path = GeodesicProbPath(scheduler=CondOTScheduler(), manifold=manifold)
optim = torch.optim.Adam(vf.parameters(), lr=LR)

# --- 4. Robust Helper Function ---
def get_text_batch(iterator, loader, vocab_size):
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(loader)
        batch = next(iterator)

    # Handle both Dict and Tensor cases automatically
    if isinstance(batch, dict):
        ids = list(batch.values())[0]
    else:
        ids = batch

    ids = ids.to(device)

    # Safety Check: If data > vocab, print error instead of crashing GPU
    if ids.max() >= vocab_size:
        raise ValueError(f"CRITICAL ERROR: Data index {ids.max()} > Vocab Size {vocab_size}. Restart Runtime.")

    x = F.one_hot(ids, num_classes=vocab_size).float()
    return x.view(x.shape[0], -1), iterator

# --- 5. Training Loop ---
iterations = 25000
print_every = 200
# --- Setup Scheduler ---
# Cosine Annealing is excellent for these types of models.
# It starts at 1e-3 and smoothly drops to 0 by the last iteration.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=iterations)

print(f"Starting Long Training (SeqLen={SEQ_LEN}, Iters={iterations})...")
start_time = time.time()
vf.train()
optim.zero_grad()

for i in range(iterations):
    # 1. Get Data
    x_1, train_iter = get_text_batch(train_iter, train_loader, vocab_size)

    # 2. Project Data to Sphere
    x_1 = x_1 / x_1.norm(dim=-1, keepdim=True)

    # 3. Sample Noise (x_0)
    x_0 = torch.randn_like(x_1).to(device)
    x_0 = x_0 / x_0.norm(dim=-1, keepdim=True)

    # 4. Sample Time
    t = torch.rand(x_1.shape[0]).to(device)

    # 5. Flow Matching Step
    path_sample = path.sample(t=t, x_0=x_0, x_1=x_1)
    pred_v = vf(path_sample.x_t, path_sample.t)

    loss = torch.pow(pred_v - path_sample.dx_t, 2).mean()

    # 6. Optimize
    loss.backward()
    torch.nn.utils.clip_grad_norm_(vf.parameters(), max_norm=1.0)
    optim.step()
    optim.zero_grad()

    # 7. Step the Scheduler
    scheduler.step()

    # 8. Logging & Saving
    if (i+1) % print_every == 0:
        elapsed = time.time() - start_time
        current_lr = scheduler.get_last_lr()[0]
        print(f'| iter {i+1:6d} | {elapsed*1000/print_every:5.2f} ms/step | loss {loss.item()*1000:8.4f} | lr {current_lr:.2e}')
        start_time = time.time()


    if (i+1) % 2000 == 0:
        save_path = f"{save_folder}/flow_model_iter_{i+1}.pt"
        torch.save(vf.state_dict(), save_path)
        print(f"✅ Safe Checkpoint saved: {save_path}")

✅ Vocab Size: 65
Starting Long Training (SeqLen=128, Iters=25000)...
| iter    200 | 384.73 ms/step | loss   0.2882 | lr 1.00e-03
| iter    400 | 381.14 ms/step | loss   0.2841 | lr 9.99e-04
| iter    600 | 382.14 ms/step | loss   0.2888 | lr 9.99e-04
| iter    800 | 382.29 ms/step | loss   0.2514 | lr 9.97e-04
| iter   1000 | 382.46 ms/step | loss   0.2417 | lr 9.96e-04
| iter   1200 | 382.49 ms/step | loss   0.2390 | lr 9.94e-04
| iter   1400 | 382.37 ms/step | loss   0.2332 | lr 9.92e-04
| iter   1600 | 382.33 ms/step | loss   0.2173 | lr 9.90e-04
| iter   1800 | 382.64 ms/step | loss   0.1965 | lr 9.87e-04
| iter   2000 | 382.73 ms/step | loss   0.1888 | lr 9.84e-04
✅ Safe Checkpoint saved: /content/drive/My Drive/Shakespeare_Flow_Models/flow_model_iter_2000.pt
| iter   2200 | 383.37 ms/step | loss   0.1597 | lr 9.81e-04
| iter   2400 | 382.48 ms/step | loss   0.1393 | lr 9.77e-04
| iter   2600 | 382.44 ms/step | loss   0.1279 | lr 9.74e-04
| iter   2800 | 382.33 ms/step | loss   0

#### Sample from trained model

In [16]:
class WrappedModel(ModelWrapper):
    def forward(self, x: torch.Tensor, t: torch.Tensor, **extras):
        return self.model(x=x, t=t)

wrapped_vf = WrappedModel(vf)

In [17]:
import torch

from transformers import GPT2TokenizerFast

# --- 1. Setup & Dimensions ---
target_seq_len = conf.model.length
target_vocab = vocab_size
desired_batch_size = 5
flat_dim = target_seq_len * target_vocab # Approx 3.2 Million

#desired_batch_size = 5
target_seq_len = conf.model.length
flat_dim = target_seq_len * vocab_size # Approx 3.2 Million

# --- 2. Generate Noise ---
print(f"Generating noise for {desired_batch_size} samples...")
x_init = torch.randn((desired_batch_size, flat_dim), device=device)
x_init = x_init / x_init.norm(dim=-1, keepdim=True)

# --- 3. Inference Wrapper ---
def inference_model(t, x):
    if t.ndim == 0:
        t = torch.full((x.shape[0],), t.item(), device=x.device)
    return vf(x, t)

# --- 4. Run ODE Solver ---
step_size = 0.01
N = 100

solver = RiemannianODESolver(velocity_model=inference_model, manifold=manifold)

print("Sampling...")
# return_intermediates=False means 'sol' will be the final Batch [5, 3.2M]
sol = solver.sample(
    x_init=x_init,
    step_size=step_size,
    method="euler",
    return_intermediates=False,
    time_grid=torch.linspace(0, 1, N).to(device),
    verbose=True,
)

# --- 5. Simple Decoding (Fixed) ---
# DO NOT use sol[-1]. sol is already the batch.
final_x = sol

# Reshape directly to [Batch, Seq, Vocab]
# We use -1 for Vocab size to handle padding differences (e.g. 50257 vs 50304)
final_x = final_x.view(desired_batch_size, target_seq_len, -1)

print(f"Debug: Final tensor shape: {final_x.shape}")
# Expected: [5, 64, 50304]

# Fisher FM on Sphere: Probabilities = Square(Amplitude)
# We just take argmax to find the most likely word
predicted_ids = torch.square(final_x).argmax(dim=-1)

# --- 6. Print Results ---
# tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

print("\n--- Generated Samples ---")
for i, sent_ids in enumerate(predicted_ids):
    text = "".join([itos[idx.item()] for idx in sent_ids])
    print(f"Sample {i+1}: {text}\n")

Generating noise for 5 samples...
Sampling...


100%|██████████| 100/100 [00:00<00:00, 105.87it/s]

Debug: Final tensor shape: torch.Size([5, 128, 65])

--- Generated Samples ---
Sample 1: wishout hos b ch morOYRK:
Ielalinge entahis gmant wample biast is mat th
hed if re ear.

weart I: l mormy maste friernchis in th

Sample 2: ? that ruol, hak yor may, haild
Aud poind mi, this, shel's yellt,pojs fry, 'ot nct negar
HAl, o, witls Thith , aeraved ne of hor

Sample 3:  what ake, ways hene ave heareI whas ruc hin cmand ds.

A kint essCqROE:
If menere cou plow I'd yy thope
en la
tor wland 
ff duc

Sample 4: l not bely of in this tean grwat'd, that aly wiscin sro
gnt pre! ant here baytifeellE:m,
The smerde bee heontth, han teecto naty

Sample 5: o, ereaF Onct.
Ahen! This wir, h hom in IVI prit
g gain thicht wriubith right will tst?
That it no fit maor et. Cowns anwitn gue



In [18]:
# --- Solver Comparison Experiment ---
print("⚔️ SOLVER SHOOTOUT: Euler vs RK4 vs Midpoint ⚔️")

# We will test 3 solvers at low step counts to see which breaks first
configs = [
    ("euler", 100), # The Baseline (High steps, simple)
    ("euler", 20),  # Can it survive with few steps?
    ("rk4", 20),    # Can a smarter solver beat Euler at low steps?
    ("midpoint", 20)
]

# Fixed noise so comparison is fair
fixed_noise = torch.randn((1, flat_dim), device=device)
fixed_noise = fixed_noise / fixed_noise.norm(dim=-1, keepdim=True)

for method, steps in configs:
    print(f"\n--- Testing {method.upper()} with {steps} Steps ---")

    # Define Solver
    # Note: Ensure your library supports 'rk4' or 'midpoint'.
    # If not, it will default to Euler, so check your library docs!
    try:
        sol = solver.sample(
            x_init=fixed_noise,
            step_size=1.0/steps, # Step size = Total Time / Steps
            method=method,
            return_intermediates=False,
            time_grid=torch.linspace(0, 1, steps).to(device),
            verbose=False
        )

        # Decode
        out = sol.view(1, target_seq_len, target_vocab)
        pred = torch.square(out).argmax(dim=-1).flatten()
        text = "".join([itos[i.item()] for i in pred])
        print(f"RESULT: {text[:100]}...") # Print first 100 chars

    except Exception as e:
        print(f"FAILED: {e}")

⚔️ SOLVER SHOOTOUT: Euler vs RK4 vs Midpoint ⚔️

--- Testing EULER with 100 Steps ---
RESULT: youn you tom patreNo. Whanve hould and;
DRNBEYS:
Ahas, Se fither wee in I letulere a
lose hath woth ...

--- Testing EULER with 20 Steps ---
RESULT: woun you tom pagreNoA,ULawve hould and;
DiNBEYS:
Ahao, Se fither wee im d letulere a
lose hath yet
 ...

--- Testing RK4 with 20 Steps ---
RESULT: youn you tom patreNo. Whanve hould and;
DRIBEYS:
Ahas, Se fither wee in diletulere a
lose hath with ...

--- Testing MIDPOINT with 20 Steps ---
RESULT: youn you tom patreNo. Whawve hould and;
DRIBESS:
Ahas, Se fither wee in diletulere a
Hose hall woth ...


### Visualize the path